# Notebook-first application walkthrough

**Problem / objective:** Turn high-volume clickstream events into behavioural, funnel and conversion features using distributed Spark operations.

**Decision / solution:** Identify where users drop out of the journey and which behavioural signals deserve product or marketing attention.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'pyspark_clickstream_analytics'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Identify where users drop out of the journey and which behavioural signals deserve product or marketing attention.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# 06 — Clickstream Analysis with PySpark

**Goal:** use distributed Spark transformations to analyse real e-commerce journeys, quantify funnel drop-off and navigation patterns, test the pipeline at one-million-row scale, and build a leakage-aware purchase-conversion model.

[Open in Google Colab](https://colab.research.google.com/github/Jorgoluka100/uni_projects/blob/main/06_Clickstream_Analysis_with_PySpark.ipynb)

This notebook separates three claims carefully:

1. **165,474 genuine click events / 24,026 genuine sessions** drive behavioural findings.
2. A **clearly labelled replicated one-million-row table** is used only for Spark load testing.
3. A separate official UCI dataset with 12,330 sessions and an observed `Revenue` outcome drives purchase-conversion modelling.

No synthetic row is presented as a unique shopper or used to inflate business metrics.

## Decision framing

An e-commerce growth team needs to know where sessions lose engagement, which journeys correlate with deeper browsing, and which active sessions should receive help or merchandising interventions. Because the event dataset does not record purchases, its funnel measures **engagement**, not sales. Purchase modelling uses the second dataset that actually contains a revenue label.

In [1]:
# Colab setup. The import check avoids reinstalling on repeated runs.
import importlib.util, subprocess, sys
if importlib.util.find_spec('pyspark') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','pyspark==3.5.1'])

import json, os, random, shutil, ssl, urllib.request, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array

SEED=42; random.seed(SEED); np.random.seed(SEED)
ROOT=Path('/content/clickstream_project' if Path('/content').exists() else './clickstream_project'); ROOT.mkdir(parents=True,exist_ok=True)
os.environ.setdefault('SPARK_LOCAL_IP','127.0.0.1')
spark=(SparkSession.builder.master('local[*]').appName('ClickstreamAnalysis').config('spark.sql.shuffle.partitions','8').config('spark.ui.enabled','false').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print({'spark':spark.version,'python':sys.version.split()[0],'seed':SEED})

{'spark': '3.5.1', 'python': '3.12.13', 'seed': 42}


## 1. Download two real, openly licensed datasets

- [UCI Clickstream Data for Online Shopping](https://archive.ics.uci.edu/dataset/553/clickstream%2Bdata%2Bfor%2Bonline%2Bshopping): 165,474 sequential clothing-store clicks, April–August 2008, CC BY 4.0, DOI `10.24432/C5QK7X`.
- [UCI Online Shoppers Purchasing Intention](https://archive.ics.uci.edu/dataset/468/online%2Bshoppers%2Bpurchasing%2Bintention%2Bdataset): 12,330 independent sessions, including 1,908 sessions ending in revenue, CC BY 4.0.

The data are historical and suitable for portfolio education; they are not evidence of current customer behaviour.

In [2]:
URLS={
 'clicks':'https://archive.ics.uci.edu/static/public/553/clickstream%2Bdata%2Bfor%2Bonline%2Bshopping.zip',
 'conversion':'https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip'}
def download_zip(url,name):
    target=ROOT/f'{name}.zip'
    if not target.exists():
        try: urllib.request.urlretrieve(url,target)
        except Exception:
            ctx=ssl._create_unverified_context();
            with urllib.request.urlopen(url,context=ctx) as r,open(target,'wb') as f: shutil.copyfileobj(r,f)
    out=ROOT/name; out.mkdir(exist_ok=True)
    with zipfile.ZipFile(target) as z: z.extractall(out)
    return out
click_dir=download_zip(URLS['clicks'],'clicks'); conversion_dir=download_zip(URLS['conversion'],'conversion')
print([p.name for p in click_dir.iterdir()],[p.name for p in conversion_dir.iterdir()])

['e-shop clothing 2008 data description.txt', 'e-shop clothing 2008.csv'] ['online_shoppers_intention.csv']


## 2. Spark ingestion and data contract

In [3]:
click_path=str(next(click_dir.glob('*.csv')))
events=(spark.read.option('header',True).option('sep',';').option('inferSchema',True).csv(click_path)
 .select(
  F.col('year').cast('int'),F.col('month').cast('int'),F.col('day').cast('int'),
  F.col('order').cast('int').alias('click_order'),F.col('country').cast('int'),
  F.col('session ID').cast('long').alias('session_id'),
  F.col('page 1 (main category)').cast('int').alias('main_category'),
  F.col('page 2 (clothing model)').alias('product_id'),F.col('colour').cast('int').alias('colour'),
  F.col('location').cast('int').alias('location'),F.col('model photography').cast('int').alias('photography'),
  F.col('price').cast('double'),F.col('price 2').cast('int').alias('above_category_average'),F.col('page').cast('int'))
 .withColumn('event_date',F.make_date('year','month','day')).cache())

row_count=events.count(); session_count=events.select('session_id').distinct().count()
null_count=events.select(sum(F.col(c).isNull().cast('int') for c in events.columns).alias('nulls')).first()['nulls']
duplicate_keys=events.groupBy('session_id','click_order').count().filter('count > 1').count()
contract={'events':row_count,'sessions':session_count,'columns':len(events.columns),'null_cells':null_count,'duplicate_session_order_keys':duplicate_keys,'date_min':str(events.agg(F.min('event_date')).first()[0]),'date_max':str(events.agg(F.max('event_date')).first()[0])}
print(json.dumps(contract,indent=2)); events.printSchema()
assert row_count==165474 and session_count==24026 and null_count==0 and duplicate_keys==0

{
  "events": 165474,
  "sessions": 24026,
  "columns": 15,
  "null_cells": 0,
  "duplicate_session_order_keys": 0,
  "date_min": "2008-04-01",
  "date_max": "2008-08-13"
}
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- click_order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: long (nullable = true)
 |-- main_category: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- photography: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- above_category_average: integer (nullable = true)
 |-- page: integer (nullable = true)
 |-- event_date: date (nullable = true)



## 3. Session modelling and engagement funnel

The source has click sequence but no cart or purchase event. The defensible funnel is therefore session depth: all sessions → at least 2 clicks → at least 5 clicks → at least 10 clicks. Calling the final stage a purchase would be false.

In [4]:
sessions=(events.groupBy('session_id').agg(
 F.count('*').alias('clicks'),F.countDistinct('product_id').alias('unique_products'),
 F.countDistinct('main_category').alias('categories_viewed'),F.avg('price').alias('avg_viewed_price'),
 F.max('page').alias('max_page'),F.min('event_date').alias('session_date'),F.first('country').alias('country'))
 .withColumn('depth_segment',F.when(F.col('clicks')>=10,'10+ clicks').when(F.col('clicks')>=5,'5-9 clicks').when(F.col('clicks')>=2,'2-4 clicks').otherwise('1 click')).cache())
funnel=sessions.agg(
 F.count('*').alias('All sessions'),
 F.sum((F.col('clicks')>=2).cast('int')).alias('2+ clicks'),
 F.sum((F.col('clicks')>=5).cast('int')).alias('5+ clicks'),
 F.sum((F.col('clicks')>=10).cast('int')).alias('10+ clicks')).toPandas().T.reset_index()
funnel.columns=['stage','sessions']; funnel['rate_from_start']=funnel.sessions/funnel.sessions.iloc[0]; funnel['drop_from_previous']=1-funnel.sessions/funnel.sessions.shift(1); funnel.loc[0,'drop_from_previous']=0
display(funnel)
fig,ax=plt.subplots(figsize=(8,4)); sns.barplot(data=funnel,x='stage',y='sessions',ax=ax,color='#3066BE'); ax.set(title='Engagement funnel (not a purchase funnel)',ylabel='Distinct real sessions'); plt.show()
depth=sessions.select('clicks').toPandas(); print(depth.clicks.describe(percentiles=[.5,.75,.9,.95,.99]).round(2))

          stage  sessions  rate_from_start  drop_from_previous
0  All sessions     24026         1.000000            0.000000
1     2+ clicks     18984         0.790144            0.209856
2     5+ clicks     11007         0.458129            0.420196
3    10+ clicks      5212         0.216932            0.526483
count    24026.00
mean         6.89
std          9.00
min          1.00
50%          4.00
75%          8.00
90%         16.00
95%         23.00
99%         42.00
max        195.00
Name: clicks, dtype: float64


## 4. Spark SQL funnel and segment diagnostics

In [5]:
events.createOrReplaceTempView('events'); sessions.createOrReplaceTempView('sessions')
monthly=spark.sql('''SELECT month(session_date) AS month, COUNT(*) AS sessions,
 ROUND(AVG(clicks),2) AS avg_clicks,
 ROUND(100*AVG(CASE WHEN clicks>=10 THEN 1 ELSE 0 END),2) AS deep_engagement_pct
 FROM sessions GROUP BY month(session_date) ORDER BY month''').toPandas()
category=spark.sql('''SELECT main_category, COUNT(*) AS clicks, COUNT(DISTINCT session_id) AS sessions,
 ROUND(AVG(price),2) AS average_viewed_price
 FROM events GROUP BY main_category ORDER BY clicks DESC''').toPandas()
display(monthly); display(category)
fig,axs=plt.subplots(1,2,figsize=(12,4)); sns.lineplot(data=monthly,x='month',y='avg_clicks',marker='o',ax=axs[0]); sns.barplot(data=category,x='main_category',y='clicks',ax=axs[1]); axs[0].set_title('Average session depth by month'); axs[1].set_title('Clicks by product category'); plt.tight_layout(); plt.show()

   month  sessions  avg_clicks  deep_engagement_pct
0      4      6941        6.94                22.09
1      5      5206        6.85                21.72
2      6      4884        6.60                20.33
3      7      5071        6.95                21.79
4      8      1924        7.35                23.39
   main_category  clicks  sessions  average_viewed_price
0              1   49742     13082                 46.71
1              4   38747      9192                 36.23
2              3   38577     10724                 40.29
3              2   38408     11490                 51.19


## 5. Journey transitions with Spark windows

`lead` is partitioned by session before computing the next category. This prevents the last click of one shopper being joined to the first click of another—a common clickstream bug.

In [6]:
w=Window.partitionBy('session_id').orderBy('click_order')
journeys=(events.withColumn('next_category',F.lead('main_category').over(w))
 .filter(F.col('next_category').isNotNull()))
transitions=(journeys.groupBy('main_category','next_category').count()
 .withColumn('from_total',F.sum('count').over(Window.partitionBy('main_category')))
 .withColumn('transition_rate',F.col('count')/F.col('from_total'))
 .orderBy(F.desc('count')))
transition_count=journeys.count(); expected_transitions=row_count-session_count
display(transitions.limit(12).toPandas())
matrix=(transitions.groupBy('main_category').pivot('next_category').agg(F.first('transition_rate')).fillna(0).orderBy('main_category').toPandas().set_index('main_category'))
fig,ax=plt.subplots(figsize=(6,5)); sns.heatmap(matrix,annot=True,fmt='.2f',cmap='Blues',ax=ax); ax.set(title='Category-to-category transition rates',xlabel='Next category',ylabel='Current category'); plt.show()
assert transition_count==expected_transitions

    main_category  next_category  count  from_total  transition_rate
0               1              1  34598       43071         0.803278
1               4              4  28354       32990         0.859473
2               3              3  26326       33035         0.796912
3               2              2  25645       32352         0.792687
4               1              3   3206       43071         0.074435
5               1              2   3153       43071         0.073205
6               2              3   2997       32352         0.092637
7               3              4   2734       33035         0.082761
8               3              1   2252       33035         0.068170
9               1              4   2114       43071         0.049082
10              2              1   2027       32352         0.062655
11              3              2   1723       33035         0.052157


## 6. Honest one-million-row Spark load test

The following table repeats source events under a `benchmark_replica` identifier and caps the result at exactly 1,000,000 rows. It tests partitioned Spark aggregations only. It is never used for shopper counts, funnel rates or model evaluation.

In [7]:
benchmark=(events.crossJoin(spark.range(7).withColumnRenamed('id','benchmark_replica')).limit(1_000_000).repartition(8,'benchmark_replica').cache())
benchmark_rows=benchmark.count()
benchmark_check=(benchmark.groupBy('benchmark_replica').agg(F.count('*').alias('rows'),F.avg('price').alias('avg_price')).orderBy('benchmark_replica'))
display(benchmark_check.toPandas()); print({'benchmark_rows':benchmark_rows,'partitions':benchmark.rdd.getNumPartitions(),'source_unique_events':row_count})
assert benchmark_rows==1_000_000

   benchmark_replica    rows  avg_price
0                  0  142858  43.772123
1                  1  142857  43.772094
2                  2  142857  43.772094
3                  3  142857  43.772094
4                  4  142857  43.772094
5                  5  142857  43.772094
6                  6  142857  43.772094
{'benchmark_rows': 1000000, 'partitions': 8, 'source_unique_events': 165474}


## 7. Real purchase-conversion modelling with Spark ML

The event dataset has no outcome, so modelling uses the UCI Online Shoppers dataset. `PageValues` is deliberately excluded because it is calculated using completed transactions and would make a real-time conversion claim vulnerable to target leakage. The deterministic hash split keeps train, validation and test records disjoint.

In [8]:
conversion_path=str(next(conversion_dir.glob('*.csv')))
buyers=spark.read.option('header',True).option('inferSchema',True).csv(conversion_path)
buyers=(buyers.withColumn('label',F.col('Revenue').cast('int')).withColumn('WeekendInt',F.col('Weekend').cast('int'))
 .withColumn('row_hash',F.pmod(F.xxhash64(*[F.col(c) for c in buyers.columns]),F.lit(100))))
buyers_count=buyers.count(); buyer_positive=buyers.filter('label=1').count()
train=buyers.filter('row_hash < 70'); validation=buyers.filter('row_hash >= 70 AND row_hash < 85'); test=buyers.filter('row_hash >= 85')
print({'rows':buyers_count,'purchases':buyer_positive,'conversion_rate':buyer_positive/buyers_count,'train':train.count(),'validation':validation.count(),'test':test.count()})

categorical=['Month','VisitorType']
numeric=['Administrative','Administrative_Duration','Informational','Informational_Duration','ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','SpecialDay','OperatingSystems','Browser','Region','TrafficType','WeekendInt']
indexers=[StringIndexer(inputCol=c,outputCol=c+'_idx',handleInvalid='keep') for c in categorical]
encoder=OneHotEncoder(inputCols=[c+'_idx' for c in categorical],outputCols=[c+'_ohe' for c in categorical],handleInvalid='keep')
assembler=VectorAssembler(inputCols=numeric+[c+'_ohe' for c in categorical],outputCol='features',handleInvalid='keep')
gbt=GBTClassifier(labelCol='label',featuresCol='features',maxIter=50,maxDepth=5,stepSize=.05,subsamplingRate=.8,seed=SEED)
pipeline=Pipeline(stages=indexers+[encoder,assembler,gbt]); fitted=pipeline.fit(train)

def scored(frame): return fitted.transform(frame).withColumn('score',vector_to_array('probability')[1])
val_scored=scored(validation).cache(); test_scored=scored(test).cache()
pr_eval=BinaryClassificationEvaluator(labelCol='label',rawPredictionCol='rawPrediction',metricName='areaUnderPR')
roc_eval=BinaryClassificationEvaluator(labelCol='label',rawPredictionCol='rawPrediction',metricName='areaUnderROC')
threshold_rows=[]
for threshold in np.arange(.10,.81,.05):
    r=(val_scored.withColumn('pred',(F.col('score')>=float(threshold)).cast('int')).agg(
      F.sum(((F.col('pred')==1)&(F.col('label')==1)).cast('int')).alias('tp'),F.sum(((F.col('pred')==1)&(F.col('label')==0)).cast('int')).alias('fp'),F.sum(((F.col('pred')==0)&(F.col('label')==1)).cast('int')).alias('fn')).first())
    precision=r.tp/max(r.tp+r.fp,1); recall=r.tp/max(r.tp+r.fn,1); f1=2*precision*recall/max(precision+recall,1e-12)
    threshold_rows.append((float(threshold),precision,recall,f1))
threshold_table=pd.DataFrame(threshold_rows,columns=['threshold','precision','recall','f1'])
selected=float(threshold_table.loc[threshold_table.f1.idxmax(),'threshold']); display(threshold_table.round(3))
print({'validation_selected_threshold':selected})

{'rows': 12330, 'purchases': 1908, 'conversion_rate': 0.15474452554744525, 'train': 8597, 'validation': 1932, 'test': 1801}
    threshold  precision  recall     f1
0        0.10      0.225   0.869  0.357
1        0.15      0.267   0.769  0.396
2        0.20      0.301   0.610  0.403
3        0.25      0.344   0.514  0.412
4        0.30      0.364   0.379  0.372
5        0.35      0.430   0.297  0.351
6        0.40      0.444   0.203  0.279
7        0.45      0.466   0.166  0.244
8        0.50      0.486   0.117  0.189
9        0.55      0.467   0.072  0.125
10       0.60      0.458   0.038  0.070
11       0.65      1.000   0.014  0.027
12       0.70      1.000   0.003  0.007
13       0.75      1.000   0.003  0.007
14       0.80      1.000   0.003  0.007
{'validation_selected_threshold': 0.25000000000000006}


## 8. Untouched conversion test and feature diagnostics

In [9]:
test_final=test_scored.withColumn('pred',(F.col('score')>=selected).cast('int')).cache()
cm=(test_final.agg(
 F.sum(((F.col('pred')==1)&(F.col('label')==1)).cast('int')).alias('tp'),F.sum(((F.col('pred')==1)&(F.col('label')==0)).cast('int')).alias('fp'),
 F.sum(((F.col('pred')==0)&(F.col('label')==1)).cast('int')).alias('fn'),F.sum(((F.col('pred')==0)&(F.col('label')==0)).cast('int')).alias('tn')).first())
precision=cm.tp/(cm.tp+cm.fp); recall=cm.tp/(cm.tp+cm.fn); f1=2*precision*recall/(precision+recall)
test_metrics={'PR_AUC':pr_eval.evaluate(test_scored),'ROC_AUC':roc_eval.evaluate(test_scored),'precision':precision,'recall':recall,'F1':f1,'alert_rate':(cm.tp+cm.fp)/(cm.tp+cm.fp+cm.fn+cm.tn),'threshold':selected}
print(json.dumps(test_metrics,indent=2))
cm_pdf=pd.DataFrame([[cm.tn,cm.fp],[cm.fn,cm.tp]],index=['Actual no purchase','Actual purchase'],columns=['Predicted no purchase','Predicted purchase'])
fig,ax=plt.subplots(figsize=(5,4)); sns.heatmap(cm_pdf,annot=True,fmt='g',cmap='Blues',ax=ax); ax.set_title('Untouched conversion test'); plt.show()

tree=fitted.stages[-1]; metadata=test_scored.schema['features'].metadata['ml_attr']['attrs']; attrs=sorted(sum(metadata.values(),[]),key=lambda x:x['idx']); feature_names=[a['name'] for a in attrs]
importance=pd.DataFrame({'feature':feature_names,'importance':tree.featureImportances.toArray()}).sort_values('importance',ascending=False)
display(importance.head(12)); fig,ax=plt.subplots(figsize=(8,5)); sns.barplot(data=importance.head(10),y='feature',x='importance',ax=ax); ax.set_title('GBT feature importance (association, not causation)'); plt.show()

{
  "PR_AUC": 0.3511953051625493,
  "ROC_AUC": 0.7628958984437946,
  "precision": 0.3433583959899749,
  "recall": 0.47735191637630664,
  "F1": 0.39941690962099125,
  "alert_rate": 0.2215435868961688,
  "threshold": 0.25000000000000006
}
                    feature  importance
7                 ExitRates    0.153103
5   ProductRelated_Duration    0.115639
1   Administrative_Duration    0.099650
4            ProductRelated    0.094766
0            Administrative    0.069085
12              TrafficType    0.060415
6               BounceRates    0.049556
9          OperatingSystems    0.037904
3    Informational_Duration    0.036981
18            Month_ohe_Oct    0.034020
10                  Browser    0.033619
2             Informational    0.032934


## 9. Persist and reload the Spark pipeline

In [10]:
model_path=ROOT/'conversion_pipeline'
if model_path.exists(): shutil.rmtree(model_path)
fitted.write().overwrite().save(str(model_path)); reloaded=PipelineModel.load(str(model_path))
original_scores=[r.score for r in test_scored.orderBy('row_hash').select('score').limit(25).collect()]
reloaded_scores=[r.score for r in reloaded.transform(test.orderBy('row_hash').limit(25)).withColumn('score',vector_to_array('probability')[1]).select('score').collect()]
reload_delta=float(np.max(np.abs(np.array(original_scores)-np.array(reloaded_scores))))
print({'pipeline_path':str(model_path),'max_prediction_delta':reload_delta}); assert reload_delta<1e-12

{'pipeline_path': 'clickstream_project/conversion_pipeline', 'max_prediction_delta': 0.0}


## 10. Acceptance tests, governance and CV evidence

In [11]:
conversion_rate=buyer_positive/buyers_count
checks={
 'real_event_count':row_count==165474,
 'real_session_count':session_count==24026,
 'click_keys_unique':duplicate_keys==0,
 'engagement_funnel_monotonic':funnel.sessions.is_monotonic_decreasing,
 'window_transitions_session_safe':transition_count==row_count-session_count,
 'million_row_benchmark_labelled':benchmark_rows==1_000_000,
 'conversion_rows_real':buyers_count==12330 and buyer_positive==1908,
 'page_values_excluded':'PageValues' not in numeric,
 'test_pr_auc_beats_prevalence':test_metrics['PR_AUC']>conversion_rate,
 'test_metrics_finite':all(np.isfinite(list(test_metrics.values()))),
 'pipeline_reload_exact':reload_delta<1e-12}
display(pd.Series(checks,name='passed').to_frame()); assert all(checks.values())

summary={'unique_click_events':int(row_count),'unique_click_sessions':int(session_count),'one_million_row_load_test':int(benchmark_rows),'engagement_funnel':{r.stage:{'sessions':int(r.sessions),'rate_from_start':float(r.rate_from_start)} for r in funnel.itertuples()},'purchase_dataset_sessions':int(buyers_count),'purchase_dataset_conversions':int(buyer_positive),'conversion_test':{k:float(v) for k,v in test_metrics.items()},'pipeline_reload_delta':reload_delta}
print('RUN-DERIVED SUMMARY'); print(json.dumps(summary,indent=2))
print(f"CV bullet: Built a PySpark clickstream pipeline over {row_count:,} real events and {session_count:,} sessions, plus a clearly labelled 1,000,000-row replicated load test; engineered session funnels and windowed journeys, then delivered a leakage-aware conversion model with {test_metrics['PR_AUC']:.3f} test PR-AUC and {test_metrics['recall']:.1%} recall.")

model_card={'system':'Clickstream analytics and conversion prioritisation','intended_use':'portfolio demonstration, exploratory merchandising and human-reviewed outreach prioritisation','not_for':'automatic pricing, exclusion or claims that engagement equals purchase','data':['UCI Clickstream Data for Online Shopping, 2008','UCI Online Shoppers Purchasing Intention'],'metrics':summary,'limitations':['event source has no purchase event, so its funnel measures engagement only','datasets are historical and may not represent current customers','load-test rows are replicated and excluded from business metrics','observational feature importance is not causal uplift','deployment requires current consented events, privacy review, drift monitoring and an experiment']}
print(json.dumps(model_card,indent=2)); spark.stop()

                                 passed
real_event_count                   True
real_session_count                 True
click_keys_unique                  True
engagement_funnel_monotonic        True
window_transitions_session_safe    True
million_row_benchmark_labelled     True
conversion_rows_real               True
page_values_excluded               True
test_pr_auc_beats_prevalence       True
test_metrics_finite                True
pipeline_reload_exact              True
RUN-DERIVED SUMMARY
{
  "unique_click_events": 165474,
  "unique_click_sessions": 24026,
  "one_million_row_load_test": 1000000,
  "engagement_funnel": {
    "All sessions": {
      "sessions": 24026,
      "rate_from_start": 1.0
    },
    "2+ clicks": {
      "sessions": 18984,
      "rate_from_start": 0.7901440106551236
    },
    "5+ clicks": {
      "sessions": 11007,
      "rate_from_start": 0.4581286939149255
    },
    "10+ clicks": {
      "sessions": 5212,
      "rate_from_start": 0.21693165737118122
    

## Conclusion

The analysis identifies engagement loss without inventing a purchase event, preserves session boundaries in navigation windows, and separates authentic behavioural evidence from a replicated scale benchmark. The conversion model is evaluated on a distinct labelled dataset with `PageValues` excluded. The next production step is to instrument consented cart and purchase events, define a canonical event schema, perform time-based backtesting, and validate any intervention through a controlled experiment.

# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

from src.analytics import build_sessions, engagement_funnel
from src.conversion import threshold_metrics

ROOT = Path(__file__).resolve().parent
EVIDENCE = ROOT / "results" / "verified_metrics.json"


def self_test() -> None:
    from pyspark.sql import SparkSession, functions as F

    spark = (
        SparkSession.builder.master("local[1]")
        .appName("clickstream-project-self-test")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    try:
        events = spark.createDataFrame(
            [
                (1, "A", 1, 10.0, 1, "2026-01-01"),
                (1, "B", 1, 12.0, 2, "2026-01-01"),
                (2, "A", 2, 9.0, 1, "2026-01-02"),
            ],
            "session_id long, product_id string, main_category int, price double, page int, event_date string",
        ).withColumn("event_date", F.to_date("event_date")).withColumn("country", F.lit(1))
        sessions = build_sessions(events)
        funnel = engagement_funnel(sessions)
        assert funnel == {"all_sessions": 2, "2+ clicks": 1, "5+ clicks": 0, "10+ clicks": 0}

        scored = spark.createDataFrame(
            [(0.1, 0), (0.4, 0), (0.8, 1), (0.9, 1)],
            "score double, label int",
        )
        metrics = threshold_metrics(scored, 0.5)
        assert metrics["precision"] == 1.0
        assert metrics["recall"] == 1.0
    finally:
        spark.stop()
    print("PySpark clickstream self-test passed.")


def check_evidence() -> None:
    report = json.loads(EVIDENCE.read_text(encoding="utf-8"))
    assert report["verification_pass"] is True
    assert report["clickstream_dataset"]["real_events"] == 165474
    assert report["clickstream_dataset"]["real_sessions"] == 24026
    assert report["load_test"]["replicated"] is True
    assert report["load_test"]["used_for_business_metrics"] is False
    assert report["conversion_dataset"]["sessions"] == 12330
    assert report["conversion_dataset"]["test_rows"] == 1801
    assert report["validation_selected_threshold"] == 0.25
    assert report["conversion_test"]["pr_auc"] > report["conversion_dataset"]["conversion_rate"]
    assert report["pipeline_reload_delta"] == 0.0
    print("Retained PySpark clickstream evidence passed.")


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--self-test", action="store_true")
    parser.add_argument("--check-evidence", action="store_true")
    args = parser.parse_args()
    if args.self_test:
        self_test()
    if args.check_evidence:
        check_evidence()
    if not args.self_test and not args.check_evidence:
        parser.error("choose --self-test or --check-evidence")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Canonical source: `src/__init__.py`


In [ ]:
"""PySpark clickstream analytics and conversion-prioritisation package."""


## Canonical source: `src/analytics.py`


In [ ]:
from __future__ import annotations


def build_sessions(events):
    """Aggregate event-level clickstream data to one row per real session."""
    from pyspark.sql import functions as F

    return (
        events.groupBy("session_id")
        .agg(
            F.count("*").alias("clicks"),
            F.countDistinct("product_id").alias("unique_products"),
            F.countDistinct("main_category").alias("categories_viewed"),
            F.avg("price").alias("avg_viewed_price"),
            F.max("page").alias("max_page"),
            F.min("event_date").alias("session_date"),
            F.first("country").alias("country"),
        )
        .withColumn(
            "depth_segment",
            F.when(F.col("clicks") >= 10, "10+ clicks")
            .when(F.col("clicks") >= 5, "5-9 clicks")
            .when(F.col("clicks") >= 2, "2-4 clicks")
            .otherwise("1 click"),
        )
    )


def engagement_funnel(sessions) -> dict[str, int]:
    """Return engagement depth; deliberately not labelled as a purchase funnel."""
    from pyspark.sql import functions as F

    row = sessions.agg(
        F.count("*").alias("all_sessions"),
        F.sum((F.col("clicks") >= 2).cast("int")).alias("two_plus"),
        F.sum((F.col("clicks") >= 5).cast("int")).alias("five_plus"),
        F.sum((F.col("clicks") >= 10).cast("int")).alias("ten_plus"),
    ).first()
    return {
        "all_sessions": int(row.all_sessions),
        "2+ clicks": int(row.two_plus),
        "5+ clicks": int(row.five_plus),
        "10+ clicks": int(row.ten_plus),
    }


def replicate_for_load_test(events, target_rows: int = 1_000_000):
    """Create clearly labelled replicated rows for Spark load testing only."""
    from pyspark.sql import functions as F

    source_rows = events.count()
    repeats = max(1, (target_rows + source_rows - 1) // source_rows)
    replicated = (
        events.crossJoin(events.sparkSession.range(repeats).withColumnRenamed("id", "replica_id"))
        .limit(target_rows)
        .withColumn("load_test_replica", F.lit(True))
    )
    return replicated


## Canonical source: `src/conversion.py`


In [ ]:
from __future__ import annotations

SEED = 42
CATEGORICAL = ["Month", "VisitorType"]
NUMERIC = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "SpecialDay",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "WeekendInt",
]
LEAKAGE_EXCLUDED = ["PageValues"]


def deterministic_split(frame):
    """Use row-content hashing so the 70/15/15 split is reproducible and disjoint."""
    from pyspark.sql import functions as F

    hashed = frame.withColumn(
        "row_hash",
        F.pmod(F.xxhash64(*[F.col(column) for column in frame.columns]), F.lit(100)),
    )
    return (
        hashed.filter("row_hash < 70"),
        hashed.filter("row_hash >= 70 AND row_hash < 85"),
        hashed.filter("row_hash >= 85"),
    )


def build_pipeline():
    from pyspark.ml import Pipeline
    from pyspark.ml.classification import GBTClassifier
    from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

    indexers = [
        StringIndexer(inputCol=column, outputCol=f"{column}_idx", handleInvalid="keep")
        for column in CATEGORICAL
    ]
    encoder = OneHotEncoder(
        inputCols=[f"{column}_idx" for column in CATEGORICAL],
        outputCols=[f"{column}_ohe" for column in CATEGORICAL],
        handleInvalid="keep",
    )
    assembler = VectorAssembler(
        inputCols=NUMERIC + [f"{column}_ohe" for column in CATEGORICAL],
        outputCol="features",
        handleInvalid="keep",
    )
    model = GBTClassifier(
        labelCol="label",
        featuresCol="features",
        maxIter=50,
        maxDepth=5,
        stepSize=0.05,
        subsamplingRate=0.8,
        seed=SEED,
    )
    return Pipeline(stages=indexers + [encoder, assembler, model])


def threshold_metrics(scored, threshold: float) -> dict[str, float]:
    from pyspark.sql import functions as F

    row = (
        scored.withColumn("pred", (F.col("score") >= float(threshold)).cast("int"))
        .agg(
            F.sum(((F.col("pred") == 1) & (F.col("label") == 1)).cast("int")).alias("tp"),
            F.sum(((F.col("pred") == 1) & (F.col("label") == 0)).cast("int")).alias("fp"),
            F.sum(((F.col("pred") == 0) & (F.col("label") == 1)).cast("int")).alias("fn"),
            F.sum(((F.col("pred") == 0) & (F.col("label") == 0)).cast("int")).alias("tn"),
        )
        .first()
    )
    precision = row.tp / max(row.tp + row.fp, 1)
    recall = row.tp / max(row.tp + row.fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    total = row.tp + row.fp + row.fn + row.tn
    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "alert_rate": float((row.tp + row.fp) / total),
    }


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 516. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is below the major-project guide and should grow only through substantive analysis/application depth.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
